# Sandbox

A scratch pad for looking at every character the **US International Scientific**
keyboard layout can produce.

The layout is read straight from `US International Scientific.klc` through the
shared parser in `tools/kbdlint`, so this notebook keeps working when the layout
changes; nothing here depends on hard-coded line numbers.

In [ ]:
import random
import sys
from collections import Counter
from pathlib import Path

REPO = Path.cwd()
while not (REPO / 'US International Scientific.klc').exists():
    if REPO.parent == REPO:
        raise SystemExit('run this notebook from inside the repository')
    REPO = REPO.parent

sys.path.insert(0, str(REPO / 'tools'))
from kbdlint import klc as klc_module

layout = klc_module.parse(REPO / 'US International Scientific.klc')
print(f'{len(layout.layout)} keys, {len(layout.dead_keys)} dead keys')

## Characters on the keys themselves

`SHIFT_STATE_NAMES` maps the MSKLC shift-state numbers to readable names:
0 is the plain key, 1 is Shift, 2 is Ctrl, 6 is AltGr and 7 is AltGr + Shift.

In [ ]:
by_state = {state: [] for state in layout.shift_states}
for row in layout.layout:
    for state, output in row.outputs.items():
        if output.code_point is not None:
            by_state[state].append(chr(output.code_point))

for state, characters in by_state.items():
    name = klc_module.SHIFT_STATE_NAMES.get(state, str(state))
    print(f'{name:>12}: {"".join(characters)}')

## Characters behind the dead keys

In [ ]:
for dead_key in layout.dead_keys:
    composites = ''.join(entry.composite_char for entry in dead_key.entries)
    print(f'U+{dead_key.root:04X} {klc_module.unicode_name(dead_key.root):<38} {composites}')

## The whole inventory

Every character the layout can produce, counted once per code point. A code point
that shows up more than once is reachable in more than one way -- that is normal
(the ISO `OEM_102` key repeats the backslash key, and several dead keys share a
default character).

In [ ]:
everything = Counter()
for row in layout.layout:
    for output in row.outputs.values():
        if output.code_point is not None:
            everything[output.code_point] += 1
for dead_key in layout.dead_keys:
    for entry in dead_key.entries:
        everything[entry.composite] += 1

print(f'{sum(everything.values())} mappings, {len(everything)} distinct characters')
print(f'highest code point: U+{max(everything):04X}')
print('reachable more than once:',
      ''.join(chr(cp) for cp, n in sorted(everything.items()) if n > 1))

In [ ]:
characters = sorted(everything)
random.shuffle(characters)
width = 30
for start in range(0, len(characters), width):
    print(''.join(chr(cp) for cp in characters[start:start + width]))

## Look up a single character

In [ ]:
def where(character):
    """Show every way `character` can be typed."""
    code_point = ord(character)
    for row in layout.layout:
        for state, output in row.outputs.items():
            if output.code_point == code_point:
                name = klc_module.SHIFT_STATE_NAMES.get(state, str(state))
                print(f'{row.virtual_key} in the {name} shift state')
    for dead_key in layout.dead_keys:
        for entry in dead_key.entries:
            if entry.composite == code_point:
                print(f'dead key U+{dead_key.root:04X} then {entry.base_char!r}')


where('ǎ')